**.enableHiveSupport()** enables integration between Spark and Hive. It allows Spark to use the Hive Metastore, create databases and persistent tables, execute Hive SQL commands, and store table metadata beyond the lifetime of a Spark session.

In [ ]:
# Spark Session
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Spark SQL")
    .master("local[*]")
    .enableHiveSupport()
    .config("spark.sql.warehouse.dir", "/data/output/spark-warehouse")
    .getOrCreate()
)

spark

In [14]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [ ]:
# Read Employee data
_schema = "first_name string, last_name string, job_title string, dob string, email string, phone string, salary double, department_id int"

emp = spark.read.format("csv").schema(_schema).option("header", True).load('/content/drive/MyDrive/employee_records_skewed.csv')

In [ ]:
# Read DEPT CSV data
_dept_schema = "department_id int, department_name string, description string, city string, state string, country string"

dept = spark.read.format("csv").schema(_dept_schema).option("header", True).load('/content/drive/MyDrive/department_data.csv')

In [ ]:
# Spark Catalog (Metadata) - in-memory/hive

spark.conf.get("spark.sql.catalogImplementation")

'hive'

In [ ]:
# Show databases
db = spark.sql("show databases")
db.show()

+---------+
|namespace|
+---------+
|  default|
+---------+



In [ ]:
spark.sql("show tables in default").show()

+---------+-------------+-----------+
|namespace|    tableName|isTemporary|
+---------+-------------+-----------+
|  default|    emp_final|      false|
|         |    dept_view|       true|
|         |emp_temp_view|       true|
|         |     emp_view|       true|
+---------+-------------+-----------+



In [ ]:
# Register dataframes are temp views

emp.createOrReplaceTempView("emp_view")

dept.createOrReplaceTempView("dept_view")

In [ ]:
# View data from table

emp_filtered = spark.sql("""
    select * from emp_view
    where department_id = 1
""")

In [ ]:
emp_filtered.show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            1|
|  Margaret|    Sutton|Administrator, ed...|1975-08-16| diana44@example.net|001-647-530-5036x...|671167.0|            1|
|      Jake|      King|       Lexicographer|1994-07-11|monica93@example.org|+1-535-652-9715x6...|702101.0|            1|
|   Heather|     Haley|         Music tutor|1981-06-01|stephanie65@examp...|   (652)815-7973x298|570960.0|            1|
|    Tracey|Washington|Travel agency man...|1986-05-07|  mark07@example.com|    001-912-206-6456|146456.0|            1|
|     Katie|  Harrison|Therapist

In [ ]:
# Create a new column dob_year and register as temp view

emp_temp = spark.sql("""
    select e.*, date_format(dob, 'yyyy') as dob_year from emp_view e
""")

In [ ]:
emp_temp.createOrReplaceTempView("emp_temp_view")

In [ ]:
spark.sql("select * from emp_temp_view").show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|dob_year|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            5|    1973|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            1|    1974|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|            2|    1990|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            3|    1968|
|  Michelle|   Elliott|      Air cabin crew|1975-03-31|tiffanyjohnston@e...|       (705)90

In [ ]:
# Join emp and dept - HINTs

emp_final = spark.sql("""
    select /*+ BROADCAST(d) */
    e.* , d.department_name
    from emp_view e left outer join dept_view d
    on e.department_id = d.department_id
""")

In [ ]:
# Show emp data

emp_final.show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|     department_name|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            5|          Hardin Inc|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            1|         Bryan-James|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|            2|Smith, Craig and ...|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            3|Pittman, Hess and...|
|  Mic

In [ ]:
# Write the data as Table

emp_final.write.format("parquet").saveAsTable("emp_final")

In [ ]:
# Read the data from Table

emp_new = spark.sql("select * from emp_final")

In [ ]:
emp_new.show()

+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------------------+
|first_name| last_name|           job_title|       dob|               email|               phone|  salary|department_id|     department_name|
+----------+----------+--------------------+----------+--------------------+--------------------+--------+-------------+--------------------+
|   Richard|  Morrison|Public relations ...|1973-05-05|melissagarcia@exa...|       (699)525-4827|512653.0|            5|          Hardin Inc|
|     Bobby|  Mccarthy|   Barrister's clerk|1974-04-25|   llara@example.net|  (750)846-1602x7458|999836.0|            1|         Bryan-James|
|    Dennis|    Norman|Land/geomatics su...|1990-06-24| jturner@example.net|    873.820.0518x825|131900.0|            2|Smith, Craig and ...|
|      John|    Monroe|        Retail buyer|1968-06-16|  erik33@example.net|    820-813-0557x624|485506.0|            3|Pittman, Hess and...|
|  Mic

In [ ]:
# Show details of metadata

spark.sql("describe extended emp_final").show()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|          first_name|              string|   NULL|
|           last_name|              string|   NULL|
|           job_title|              string|   NULL|
|                 dob|              string|   NULL|
|               email|              string|   NULL|
|               phone|              string|   NULL|
|              salary|              double|   NULL|
|       department_id|                 int|   NULL|
|     department_name|              string|   NULL|
|                    |                    |       |
|# Detailed Table ...|                    |       |
|             Catalog|       spark_catalog|       |
|            Database|             default|       |
|               Table|           emp_final|       |
|               Owner|                root|       |
|        Created Time|Sat May 23 14:00:...|       |
|         La

A Delta table stores two things:

**Data Files (Parquet)**

Actual data is stored as Parquet files on disk (S3, ADLS, DBFS etc.)

Every write operation adds new parquet files — old ones are not overwritten

**Transaction Log (_delta_log)**

A folder that stores JSON files tracking every operation ever done on the table

This is what makes Delta different from a plain Parquet table

In [ ]:
# Step 1 — Download NYC Taxi dataset (public source)
!wget -q https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet -O /content/yellow_tripdata_2023-01.parquet

# Step 2 — Start Spark Session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("NYCTaxi") \
    .getOrCreate()

# Step 3 — Simulate dbutils.fs.ls() using os.listdir()
import os

directory = "/content/"
files = os.listdir(directory)
for f in files:
    full_path = os.path.join(directory, f)
    size = os.path.getsize(full_path) if os.path.isfile(full_path) else "-"
    print(f"  name={f:<45} size={size}")

# Step 4 — Read the Parquet file
df = spark.read.parquet("/content/yellow_tripdata_2023-01.parquet")

  name=.config                                       size=-
  name=.ipynb_checkpoints                            size=-
  name=yellow_tripdata_2023-01.parquet               size=47673370
  name=sample_data                                   size=-


In [ ]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2023-01-01 00:32:10|  2023-01-01 00:40:36|            1.0|         0.97|       1.0|                 N|         161|         141|           2|        9.3|  1.0|    0.5|       0.

In [ ]:
df.count()

3066766

In [ ]:
# Check filter data
# select count(1) from nyctaxi where vendor_id = 'VTS' and trip_distance > 1.8

df.where("vendor_id = 2" and "trip_distance > 1.8").count()

1505159

With columnar formats like Parquet and ORC::

Spark reads only the selected column.

This is called Column Pruning.

Much faster than reading the entire file.

**_metadata.file_name** is a hidden Spark metadata column — reveals which Parquet file each row came from